In [1]:
import os
os.environ['ADASP_DATA'] = '/tsi'
from adasp_data_management import music, event, speaker

/home/ids/xzhuang/anaconda3/envs/fewshot_audio/lib/python3.9/site-packages/jams/__init__.py:24: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


In [2]:
db = event.BirdClef()

BirdClef: load metadata file /tsi/dcase/BirdClef/metadata.csv


In [3]:
db.pdf_metadata.iloc[0]

file_path             /tsi/dcase/BirdClef/ttahara_birdsong-resampled...
rating                                                              3.5
playback_used                                                        no
ebird_code                                                       aldfly
date                                                         2013-05-25
pitch                                                     Not specified
filename                                                   XC134874.mp3
speed                                                     Not specified
species                                                Alder Flycatcher
number_of_notes                                           Not specified
title                     XC134874 Alder Flycatcher (Empidonax alnorum)
secondary_labels      ['Empidonax minimus_Least Flycatcher', 'Leioth...
bird_seen                                                           yes
sci_name                                              Empidonax 

In [2]:
import os

def create_mirror_structure(metadata_df, mirror_dir):
    # Ensure mirror_dir is absolute
    mirror_dir = os.path.abspath(mirror_dir)
    os.makedirs(mirror_dir, exist_ok=True)
    
    for _, row in metadata_df.iterrows():
        # Get the path from your metadata
        raw_src = row['file_path']
        
        # Use realpath to get the 'true' physical path
        src_path = os.path.realpath(raw_src)
        
        # Double check visibility in Python
        if not os.path.exists(src_path):
            print(f"Python cannot see: {src_path}")
            continue

        label = row['id']
        target_dir = os.path.join(mirror_dir, label)
        os.makedirs(target_dir, exist_ok=True)
        
        # Define target file path
        target_file = os.path.join(target_dir, os.path.basename(src_path))
        
        # Clear existing symlinks to avoid 'File exists' errors
        if os.path.lexists(target_file):
            os.remove(target_file)
            
        # Create the link using the fully resolved source path
        os.symlink(src_path, target_file)

In [3]:
vox = speaker.VoxCeleb1()
vox_metadata = vox.pdf_metadata
vox_metadata.iloc[0]

VoxCeleb1: load metadata file /tsi/plato_sons/speech/VoxCeleb/VoxCeleb1/metadata.csv


file_path        /tsi/plato_sons/speech/VoxCeleb/VoxCeleb1/vox1...
id                                                         id10053
VGG_face1_id                                      Andrew_Lee_Potts
gender                                                           m
nationality                                                     UK
set                                                            dev
n_speakers                                                       1
n_channels                                                       1
sampling_rate                                                16000
duration                                                 12.400063
cutoff_freq                                                   5906
Name: 0, dtype: object

In [4]:

# --- WORKFLOW ---
# 1. Get your metadata from your private package
# metadata = your_private_package.get_metadata()

# 2. Setup the Mirror
# MIRROR_AUDIO_PATH = 'data/BirdClef_Audio_Mirror'

MIRROR_AUDIO_PATH = 'Datasets/VoxCeleb1_Mirror'
create_mirror_structure(vox_metadata, MIRROR_AUDIO_PATH)

In [16]:
vox_metadata['id'].nunique()

1251

In [10]:
# 1. Get the frequency of each speaker ID
counts = vox_metadata['id'].value_counts()

# 2. Filter for IDs where the count is less than 10
low_sample_classes = counts[counts < 10].index.tolist()

print(f"Number of classes with < 10 samples: {len(low_sample_classes)}")
print("First few classes to remove:", low_sample_classes[:5])

Number of classes with < 10 samples: 0
First few classes to remove: []
